# Final model development & verification — stepwise AFT with distribution fall-back

**Inputs** (both produced by `master_pipeline_headway_v4`)

| File | Sheet | Used for |
|---|---|---|
| `Tables/03_distribution_fits.xlsx` | `All_fits_loc0` | the per-pair distribution **AIC rank order** that the fall-back cascade walks |
| `Tables/04_covariate_screen.xlsx` | `Screening_long` | the per-pair **candidate covariate pool**, its ordering, and the marginal effects used for the marginal-vs-adjusted comparison |

**Output** `Tables/07_final_models_verified.xlsx` — GoF, publication table, coefficients,
**selection path**, **cascade attempt log**, marginal-vs-adjusted comparison, and one
native Excel Cox–Snell Q–Q chart per pair. Figures also go to `Graphics/`.

---

## The algorithm

For each pair, one master row set is built once, so every model in that pair — null,
every stepwise trial, and the final model — is fitted on identical rows and is strictly
nested.

**A. Covariate selection — forward entry with backward fall-back**

- *Enter*: fit each not-yet-included covariate conditional on those already in; add the
  one with the largest ΔAIC, provided ΔAIC ≥ `DAIC_ENTER` (2) **and** LR *p* < `ALPHA_ENTER`.
- *Fall back*: after every entry, re-test each already-selected covariate by removing it;
  drop it if ΔAIC < `DAIC_STAY` (0) **or** LR *p* ≥ `ALPHA_STAY`.
- The stay rule is deliberately **weaker** than the entry rule so a dropped covariate
  cannot immediately re-enter. A visited-set cycle guard and `MAX_STEPS` back this up.
- Every trial is warm-started from its parent model, so LR statistics along the path are
  non-negative by construction.

**B. Distribution fall-back cascade**

Distributions are tried in the Table-3 AIC order. For each, the whole of (A) is re-run and
the resulting model is verified with Cox–Snell residuals against Exp(1). `CASCADE_RULE`:

- `"first_pass"` (default) — keep the first distribution whose final model passes and converges.
- `"best_pass"` — try all `N_CASCADE`, keep the lowest-AIC model among those that pass.

If none pass, the lowest-AIC attempt is kept and flagged `UNVERIFIED`. **Every attempt is
logged**, not just the winner — the cascade is a search over distributions and the paper
has to show what was searched.

---

## Issues found in `10_1_final_model_fit_verification` and how they are fixed

**Inputs and wiring**

1. **`FINAL_SPEC` was hand-typed and had drifted from the pipeline.** It assigned
   `weibull_min` to `BTW_following_4W` and `BTW_following_MT_3W` and `gengamma` to four
   pairs; Table 3 actually ranks `gengamma`, `gamma`, `gamma`, `weibull_min`, `invgauss`, … .
   It also listed covariates (`flow`, `occupancy`) that Table 4 shows are not significant
   for those pairs, and omitted `Site` and `Off_centeredness` entirely — `Site` is the
   second-strongest covariate for `BTW_following_MT_3W` (ΔAIC 12.7) and `Off_centeredness`
   is significant for `BTW_following_4W`. *Fixed:* the spec is read from Tables 3 and 4;
   nothing is hard-coded.
2. **The flow column bug came back.** `COV["flow"] = ("Flow_pcu/hr", "cont", 1000, …)`.
   The column is `Flow_pcu/hr/m` and its range is 197–922, so the unit was wrong by an
   order of magnitude too. *Fixed:* pattern resolution, unit = per +100 pcu/hr/m.
3. **Key mismatch.** The notebook's keys (`speed`, `speed_diff`, `flow`, `occupancy`) do
   not match Table 4's `Covariate` values (`Target_Speed_km/hr`, `Speed_Difference`,
   `Flow_pcu/hr/m`, `Occupancy`, `Off_centeredness`, `Site`), so no join to the screen was
   possible. *Fixed:* the registry is rebuilt with the same rule the master pipeline uses,
   and a reconciliation report prints anything that fails to match.
4. **Wrong sheet name.** It reads `All_fits`; v4 writes `All_fits_loc0`. *Fixed:* resilient
   sheet resolution by preferred name, then by required columns.
5. **Output collision.** It wrote `06_model_fit_verification.xlsx` while the master
   pipeline owns `06_compound_models.xlsx`. *Fixed:* output is `07_final_models_verified.xlsx`.

**Model specification**

6. **The markdown and the code disagree about `loc`.** The header says "`loc` is estimated
   (shared) so the physical minimum headway is respected" and the Cell-3 banner says
   "free loc", but `_nll` hard-codes `loc=0`. The advertised protection was never
   implemented. *Fixed:* `LOC_MODE` is explicit. Default `"zero"`, because that is how
   Table 3 ranked the distributions and it keeps the model a genuine AFT. `"free"` is
   available, mapped through a logistic into (0, min t) so the threshold can never cross
   the smallest observation — but note that with a free threshold `t = loc + scale_i·Y`
   is **not** a pure AFT (the threshold does not scale with the covariates), the
   likelihood is unbounded as loc → min t for 3-parameter families, and Table 3's ranking
   no longer applies. Treat it as a sensitivity run, and the notebook warns if you set it.
7. **Shapes were forced positive** by `sh = [exp(p[i]) …]`. `gengamma`'s second shape *c*
   and `pearson3`'s skew are legitimately negative, so the exp-link silently deleted half
   of the parameter space of two families that are in the cascade. *Fixed:* shapes are
   estimated unconstrained; non-finite likelihoods are rejected by the penalty instead.
8. **No standardisation.** Flow has sd ≈ 115; `exp(b0 + b1·z)` overflows. *Fixed:*
   continuous covariates are z-scored inside the optimiser and back-transformed for reporting.

**Estimation and testing**

9. **Forward selection was cold-started at every trial.** `fit_aft` always rebuilt `x0`
   from `dist.fit`, so the larger model was not guaranteed to reach at least the smaller
   model's likelihood. When it did not, `2*(llt − ll_cur)` went negative, `chi2.sf`
   returned ≈ 1, and the covariate was rejected as useless. This is a silent
   under-selection bug and it is the most consequential one in the file. *Fixed:* every
   nested trial is warm-started from its parent.
10. **Three different null fits.** `forward_select` fitted its own null, then Cell 4
    refitted the null again after the cascade, and the reported `joint_LR` and
    `dAIC_vs_null` were computed against that third fit. They could disagree with the
    path that produced the model. *Fixed:* one null per (pair, distribution), passed in.
11. **`numeric_se` returned `sqrt(abs(diag(cov)))`.** A non-positive-definite Hessian —
    i.e. a failed optimisation — was converted into a plausible-looking SE, *z* and *p*.
    *Fixed:* `NaN` when the variance is not positive, and `None` when a Hessian
    perturbation leaves the support. The step size is now relative, not a fixed `1e-4`.
12. **No confidence intervals.** *Added*, on the % change scale.
13. **Unclamped LR.** `LR = 2*(ll − ll0)` could be negative and was reported as such.
    *Fixed:* clamped at 0, with the warm-start guarantee behind it.
14. **Listwise deletion was missing.** `g = df[df[OUTCOME].notna() & …]` did not drop rows
    with a missing covariate; those produce `NaN` in `Z`, the likelihood collapses to the
    `1e12` penalty, and the optimiser still returns numbers that get written to Excel.
    *Fixed:* one master row set per pair.

**The cascade itself**

15. **The gate maximised a p-value.** `best_seen` was chosen by the largest Cox–Snell KS
    *p*, and the cascade stopped at the first `pks >= ALPHA`. A p-value is not a model
    selection criterion. *Fixed:* passing is a gate, but the choice among passers is by
    AIC (`best_pass`) or by rank order (`first_pass`), both explicit and both logged.
16. **The Cox–Snell KS p-value is not valid as printed.** The parameters were estimated
    from the same sample, so the analytic KS *p* is biased upward — the gate is
    **lenient** and the cascade will rarely fire. *Fixed:* `CS_GATE="ks_bootstrap"`
    gives the honest parametric-bootstrap *p*; the analytic version is kept as the fast
    default and the notebook prints a warning saying which one was used. Anderson–Darling
    on the residuals and the Q–Q correlation are reported alongside.
17. **Nothing recorded what was searched.** *Added:* `Cascade_attempts` logs every
    distribution tried with its selected set, AIC and verification result;
    `Selection_path` logs every ENTER and DROP with its ΔAIC and *p*.
18. **`converged` could belong to a discarded run**, and jitter seed 0 produced no jitter.
    *Fixed.*
19. **Excel sheet names could collide.** `("qq_"+pair)[:31]` neither strips the characters
    Excel forbids (`: \ / ? * [ ]`) nor de-duplicates after truncation. *Fixed.*

**Still open — decide before submission**

- **Two-stage selection.** `CANDIDATE_RULE="all"` (default) offers every covariate to the
  stepwise search, so selection happens once and the reported *p*-values are only subject
  to the usual stepwise optimism. Setting it to `"strict"` reproduces the univariate
  pre-screen, which is a second selection stage on the same data — the *p*-values then
  become conditional on having passed stage 1. Pick one and describe it honestly.
- **Stepwise inference is optimistic.** Whatever rule you use, the CIs and *p*-values of a
  stepwise-selected model are not the CIs of a pre-specified model. The `M2_full` models
  in `06_compound_models.xlsx` are the pre-specified comparison; report both.
- **Right truncation at 5 s** and the 1/30 s discretisation of the response still apply
  here, since this notebook inherits the same likelihood.
- **`PR_following_NMT_3W` and `PR_following_4W`** have no covariate that clears entry.
  An intercept-only distribution is the honest answer for those two; do not hunt for a
  rule that manufactures one.


In [1]:
# =============================================================================
# Cell 1 — Imports, paths, configuration
# =============================================================================
import os, re, warnings, platform
from datetime import datetime

import numpy as np
import pandas as pd
from scipy import stats, optimize

warnings.simplefilter("ignore")

# ------------------------------------------------------------------ paths ---
BASE      = r"D:\Headway"
DATA_PATH = os.path.join(BASE, "data3.xlsx")
TABLES    = os.path.join(BASE, "Tables")
GRAPHICS  = os.path.join(BASE, "Graphics")
os.makedirs(TABLES,   exist_ok=True)
os.makedirs(GRAPHICS, exist_ok=True)

# INPUTS produced by the master pipeline (v4)
EXCEL03 = os.path.join(TABLES, "03_distribution_fits.xlsx")   # distribution ranking per pair
EXCEL04 = os.path.join(TABLES, "04_covariate_screen.xlsx")    # univariate covariate screen

# OUTPUT -- deliberately 07_*, so it does NOT overwrite 06_compound_models.xlsx
OUT_XLSX = os.path.join(TABLES, "07_final_models_verified.xlsx")

# --------------------------------------------------------- model definition --
OUTCOME = "Time_Headway"
STRATUM = "Pair"
EXCLUDE_FROM_COVARIATES = [OUTCOME, "V_Target", "V_Subject", "V_Leading_Class",
                           "Pair", "Leading_Speed_km/hr"]

# ------------------------------------------------- stepwise selection rules --
# ENTRY  : add the covariate with the largest conditional dAIC, provided
#          dAIC >= DAIC_ENTER  AND  LR p < ALPHA_ENTER
# STAY   : after every addition, re-test each already-selected covariate; drop it
#          if conditional dAIC < DAIC_STAY  OR  LR p >= ALPHA_STAY
# The STAY rule MUST be weaker than the ENTRY rule or the loop can cycle
# (a covariate dropped at STAY must not immediately satisfy ENTRY again).
DAIC_ENTER  = 2.0
ALPHA_ENTER = 0.05
DAIC_STAY   = 0.0
ALPHA_STAY  = 0.05
USE_BACKWARD = True          # False -> pure forward selection, no fall-back step
MAX_STEPS    = 25            # hard cap; the cycle guard normally stops first

# Which covariates are offered to the stepwise search, read from Table 4:
#   "all"     -> every screened covariate is offered; the CONDITIONAL criterion
#                decides. Single-stage selection, no double-dipping. (default)
#   "liberal" -> only covariates with improves == "Yes"        (univariate p<0.05)
#   "strict"  -> only covariates with improves_strict == "Yes" (p<0.05 AND dAIC>=2)
#   "bh"      -> only covariates with survives_BH_within_pair == "Yes"
# Note: "liberal"/"strict"/"bh" make this a two-stage screen; the reported p-values
# are then conditional on having passed stage 1 and are optimistic. Say so if used.
CANDIDATE_RULE = "all"

# ---------------------------------------------- distribution fall-back rule --
# Distributions are tried in the Table-3 AIC rank order. The first one whose FINAL
# model passes the Cox-Snell check AND converges is kept ("first_pass" = the
# fall-back cascade). "best_pass" instead keeps the lowest-AIC model among all that
# pass. If none pass, the best-AIC attempt is reported and flagged UNVERIFIED.
CASCADE_RULE = "first_pass"
N_CASCADE    = 6             # max distributions to try per pair
CS_ALPHA     = 0.05          # Cox-Snell KS pass threshold

# Cox-Snell gate. "ks_analytic" is fast but LENIENT: the KS p-value assumes the
# parameters are known, whereas they were estimated from the same sample, so it is
# biased upward and almost everything passes at rank 1 (the cascade rarely fires).
# "ks_bootstrap" is the honest gate; it costs N_BOOT_CS refits per attempt.
CS_GATE    = "ks_analytic"
N_BOOT_CS  = 200             # only used when CS_GATE == "ks_bootstrap"

# ----------------------------------------------------------- loc handling ----
# "zero" : loc = 0. Matches how Table 3 ranked the distributions, keeps the model a
#          genuine AFT (t = scale_i * Y), and keeps the LR tests regular. DEFAULT.
# "free" : one shared threshold loc in (0, min(t)), estimated on a logistic scale.
#          Physically appealing (a minimum safe headway) BUT: (i) t = loc + scale_i*Y
#          is no longer a pure AFT because the threshold does not scale; (ii) the
#          likelihood is unbounded as loc -> min(t) for 3-parameter families;
#          (iii) Table 3's ranking was computed at loc = 0, so the cascade order is
#          no longer the right order. Use only as a sensitivity run.
LOC_MODE = "zero"

# --------------------------------------------------------------- numerics ----
ALPHA           = 0.05
N_STARTS_SEARCH = 1          # warm-started, so one start is enough during the search
N_STARTS_FINAL  = 4          # more starts for the retained model
BIG_NLL         = 1e10
ETA_CLIP        = 25.0
RNG_SEED        = 20250101
MIN_BIN_GROUP   = 5
MAKE_PNG        = True       # also write matplotlib figures to the Graphics folder

print(f"python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__}")
print(f"selection: stepwise (backward fall-back = {USE_BACKWARD}) | "
      f"enter dAIC>={DAIC_ENTER}, p<{ALPHA_ENTER} | stay dAIC>={DAIC_STAY}, p<{ALPHA_STAY}")
print(f"candidates from Table 4 rule = '{CANDIDATE_RULE}' | cascade = '{CASCADE_RULE}' | "
      f"loc = '{LOC_MODE}' | Cox-Snell gate = '{CS_GATE}'")
print("output ->", OUT_XLSX)

python 3.14.6 | numpy 2.4.6 | pandas 2.3.3
selection: stepwise (backward fall-back = True) | enter dAIC>=2.0, p<0.05 | stay dAIC>=0.0, p<0.05
candidates from Table 4 rule = 'all' | cascade = 'first_pass' | loc = 'zero' | Cox-Snell gate = 'ks_analytic'
output -> D:\Headway\Tables\07_final_models_verified.xlsx


In [2]:
# =============================================================================
# Cell 2 — Load data3 and rebuild the covariate registry
#   The registry is built with EXACTLY the same rule as the master pipeline, so
#   its keys are the raw column names and therefore join 1:1 with the 'Covariate'
#   column of Table 4. The old notebook used its own short keys ('speed',
#   'flow', ...) and a hard-coded 'Flow_pcu/hr' column, so neither the join nor
#   the flow covariate could ever work.
# =============================================================================
df = pd.read_excel(DATA_PATH)
df.columns = [str(c).strip() for c in df.columns]

def find_col(patterns, cols):
    for c in cols:
        low = c.lower()
        if all(p in low for p in patterns):
            return c
    return None

SUBJECT_COL = "V_Target" if "V_Target" in df.columns else "V_Subject"
SPEED_COL   = find_col(["target", "speed"], df.columns) or find_col(["subject", "speed"], df.columns)
LEADSPD_COL = find_col(["leading", "speed"], df.columns)
FLOW_COL    = find_col(["flow"], df.columns)
SPDDIF_COL  = find_col(["speed", "differ"], df.columns)

n_raw = len(df)
df = df[np.isfinite(pd.to_numeric(df[OUTCOME], errors="coerce"))].copy()
df = df[df[OUTCOME] > 0].copy()

def _numeric_unit_label(col, s):
    rng_ = float(np.nanmax(s) - np.nanmin(s))
    if rng_ > 200: return 100.0, "per +100 units"
    if rng_ > 20:  return 10.0,  "per +10 units"
    if rng_ > 2:   return 1.0,   "per +1 unit"
    return 0.1, "per +0.10 unit"

COVARIATES = {}
for col in df.columns:
    if col in EXCLUDE_FROM_COVARIATES:
        continue
    s = df[col]
    if pd.api.types.is_bool_dtype(s):
        newc = f"_{col}_01"
        df[newc] = s.astype(int)
        COVARIATES[col] = dict(col=newc, type="bin", unit=1.0,
                               label="True vs False", ref="False")
    elif pd.api.types.is_numeric_dtype(s):
        unit, lab = _numeric_unit_label(col, s.values.astype(float))
        if col in (SPEED_COL, SPDDIF_COL):
            unit, lab = 1.0, "per +1 km/h"
        elif col == FLOW_COL:
            unit, lab = 100.0, "per +100 pcu/hr/m"
        COVARIATES[col] = dict(col=col, type="cont", unit=unit, label=lab, ref="")
    else:
        levels = sorted(pd.Series(s).dropna().astype(str).unique())
        if len(levels) == 2:
            newc = f"_{col}_01"
            df[newc] = (df[col].astype(str) == levels[1]).astype(int)
            COVARIATES[col] = dict(col=newc, type="bin", unit=1.0,
                                   label=f"{levels[1]} vs {levels[0]}", ref=levels[0])

COV_ORDER = list(COVARIATES.keys())
print(f"rows: {n_raw} -> {len(df)}")
print(f"leading speed column '{LEADSPD_COL}' is EXCLUDED (exactly collinear with "
      f"target speed + speed difference)")
print("covariate registry:")
for k, v in COVARIATES.items():
    print(f"   {k:22s} {v['type']:4s} col={v['col']:24s} unit={v['unit']:<6g} {v['label']}")

rows: 898 -> 898
leading speed column 'Leading_Speed_km/hr' is EXCLUDED (exactly collinear with target speed + speed difference)
covariate registry:
   Target_Speed_km/hr     cont col=Target_Speed_km/hr       unit=1      per +1 km/h
   Speed_Difference       cont col=Speed_Difference         unit=1      per +1 km/h
   Off_centeredness       bin  col=_Off_centeredness_01     unit=1      True vs False
   Occupancy              bin  col=_Occupancy_01            unit=1      True vs False
   Flow_pcu/hr/m          cont col=Flow_pcu/hr/m            unit=100    per +100 pcu/hr/m
   Site                   bin  col=_Site_01                 unit=1      Tikatuli vs Shahjahanpur


In [3]:
# =============================================================================
# Cell 3 — Read Table 3 (distribution ranking) and Table 4 (covariate screen)
#   Nothing is hard-coded. The old notebook carried a hand-typed FINAL_SPEC that
#   had already drifted from the pipeline output (wrong distributions, no Site,
#   no Off_centeredness) and read a sheet named 'All_fits' that v4 renamed.
# =============================================================================
def _pick_sheet(path, wanted, must_have):
    """Resolve a sheet by preferred name, else by required columns."""
    xl = pd.ExcelFile(path)
    for w in wanted:
        if w in xl.sheet_names:
            d = pd.read_excel(xl, w)
            if all(c in d.columns for c in must_have):
                return d, w
    for s in xl.sheet_names:
        d = pd.read_excel(xl, s)
        if all(c in d.columns for c in must_have):
            return d, s
    raise KeyError(f"No sheet in {os.path.basename(path)} has columns {must_have}. "
                   f"Sheets present: {xl.sheet_names}")

# ---------------------------------------------------- Table 3 -> RANKING -----
fits3, sh3 = _pick_sheet(EXCEL03,
                         ["All_fits_loc0", "All_fits"],
                         ["Pair", "Distribution", "Rank"])
if LOC_MODE == "free":
    print("WARNING: LOC_MODE='free' but the ranking in Table 3 was computed at loc=0. "
          "The cascade order is therefore not the correct AIC order for this run.")

RANKING, RANK_AIC = {}, {}
for pair, sub in fits3.groupby("Pair"):
    sub = sub.sort_values("Rank")
    names = [d for d in sub["Distribution"].tolist() if hasattr(stats, d)]
    RANKING[pair] = names
    if "AIC" in sub.columns:
        RANK_AIC[pair] = dict(zip(sub["Distribution"], sub["AIC"]))

# ------------------------------------------------- Table 4 -> CANDIDATES -----
scr4, sh4 = _pick_sheet(EXCEL04,
                        ["Screening_long"],
                        ["Pair", "Covariate", "dAIC"])
scr4["_d"] = pd.to_numeric(scr4["dAIC"], errors="coerce")

_rule_col = {"all": None, "liberal": "improves",
             "strict": "improves_strict", "bh": "survives_BH_within_pair"}[CANDIDATE_RULE]

CANDIDATES, MARGINAL = {}, {}
for pair, sub in scr4.groupby("Pair", sort=False):
    sub = sub.sort_values("_d", ascending=False)
    estimable = sub[sub.get("improves", pd.Series("n/a", index=sub.index)) != "n/a"]
    pool = estimable if len(estimable) else sub
    if _rule_col is not None and _rule_col in sub.columns:
        pool = pool[pool[_rule_col] == "Yes"]
    CANDIDATES[pair] = [c for c in pool["Covariate"].tolist() if c in COVARIATES]
    MARGINAL[pair] = {r["Covariate"]: dict(marginal_effect=r.get("Effect"),
                                           marginal_dAIC=r.get("_d"),
                                           marginal_LR_p=r.get("LR_p"))
                      for _, r in sub.iterrows()}

# ----------------------------------------------------- reconciliation --------
PAIRS = [p for p in scr4["Pair"].drop_duplicates().tolist()
         if p in RANKING and len(RANKING[p])]
missing_rank = [p for p in scr4["Pair"].unique() if p not in PAIRS]
unknown_cov  = sorted(set(scr4["Covariate"]) - set(COVARIATES))
unused_cov   = sorted(set(COVARIATES) - set(scr4["Covariate"]))

print(f"Table 3 sheet '{sh3}' -> ranking for {len(RANKING)} pairs")
print(f"Table 4 sheet '{sh4}' -> screen for {scr4['Pair'].nunique()} pairs")
if missing_rank:
    print("  !! screened but no distribution ranking, SKIPPED:", missing_rank)
if unknown_cov:
    print("  !! in Table 4 but not in the registry (check column names):", unknown_cov)
if unused_cov:
    print("  !! in the registry but not screened in Table 4:", unused_cov)

print(f"\nCandidate pools offered to the stepwise search (rule='{CANDIDATE_RULE}'):")
for p in PAIRS:
    print(f"   {p:24s} rank1={RANKING[p][0]:12s} candidates={CANDIDATES[p]}")

Table 3 sheet 'All_fits_loc0' -> ranking for 10 pairs
Table 4 sheet 'Screening_long' -> screen for 8 pairs

Candidate pools offered to the stepwise search (rule='all'):
   BTW_following_4W         rank1=gengamma     candidates=['Speed_Difference', 'Target_Speed_km/hr', 'Site', 'Off_centeredness', 'Occupancy', 'Flow_pcu/hr/m']
   BTW_following_MT_3W      rank1=gamma        candidates=['Target_Speed_km/hr', 'Speed_Difference', 'Site', 'Off_centeredness', 'Occupancy', 'Flow_pcu/hr/m']
   BTW_following_NMT_3W     rank1=gamma        candidates=['Target_Speed_km/hr', 'Site', 'Flow_pcu/hr/m', 'Off_centeredness', 'Speed_Difference', 'Occupancy']
   PR_following_MT_3W       rank1=weibull_min  candidates=['Target_Speed_km/hr', 'Occupancy', 'Speed_Difference', 'Site', 'Off_centeredness', 'Flow_pcu/hr/m']
   BTW_following_MT_2W      rank1=invgauss     candidates=['Speed_Difference', 'Target_Speed_km/hr', 'Off_centeredness', 'Site', 'Flow_pcu/hr/m', 'Occupancy']
   PR_following_NMT_3W      rank1=we

In [4]:
# =============================================================================
# Cell 4 — AFT machinery:  log(scale_i) = b0 + b' z_i ,  shape shared,  loc per LOC_MODE
#
# Parameter vector layout (m = number of covariates in the model):
#     p = [ shape_0 ... shape_{ns-1} , b0 , beta_0 ... beta_{m-1} ]  (+ [loc_raw])
# loc_raw enters only when LOC_MODE == "free" and is mapped through a logistic to
# (0, min(t)) so the threshold can never cross the smallest observation.
#
# Fixes relative to the old notebook:
#   * shapes are NOT exp()-transformed. exp() forces every shape positive, which
#     silently forbids the negative c of gengamma and the negative skew of
#     pearson3 -- two of the families in the cascade.
#   * continuous covariates are z-standardised inside the optimiser (flow has
#     sd ~ 115, so exp(b0 + b1*z) overflowed) and back-transformed for reporting.
#   * every nested model is WARM-STARTED from its parent, so a larger model can
#     never come out with a lower log-likelihood than the model it contains.
#     Without this, forward selection silently under-selects: 2*(ll1 - ll0) goes
#     negative, chi2.sf returns ~1, and the covariate is rejected as "not useful".
#   * standard errors are NaN when the observed information is not positive
#     definite. The old numeric_se did sqrt(abs(diag(cov))), which manufactures a
#     plausible-looking SE (and z, and p) out of a failed optimisation.
#   * the Hessian step is relative to each parameter's magnitude, not a fixed 1e-4.
# =============================================================================
def _loc_from_raw(raw, loc_max):
    return loc_max / (1.0 + np.exp(-np.clip(raw, -30.0, 30.0)))

def _unpack(p, m, ns, loc_max):
    shapes = p[:ns]
    b0     = p[ns]
    betas  = p[ns + 1: ns + 1 + m]
    loc    = 0.0 if loc_max is None else _loc_from_raw(p[ns + 1 + m], loc_max)
    return shapes, b0, betas, loc

def _nll(p, t, Z, dist, ns, loc_max):
    m = Z.shape[1]
    shapes, b0, betas, loc = _unpack(p, m, ns, loc_max)
    lin = b0 + (Z @ betas if m else 0.0)
    scale = np.exp(np.clip(lin, -ETA_CLIP, ETA_CLIP))
    with np.errstate(all="ignore"):
        lp = dist.logpdf(t, *shapes, loc=loc, scale=scale)
    if not np.all(np.isfinite(lp)):
        return BIG_NLL
    v = -float(np.sum(lp))
    return v if np.isfinite(v) else BIG_NLL

def _optimise(fn, x0, args, n_starts, jitter=0.15, seed=RNG_SEED):
    r0 = np.random.default_rng(seed)
    x0 = np.asarray(x0, float)
    starts = [x0] + [x0 + r0.normal(0, jitter, size=len(x0)) for _ in range(max(0, n_starts - 1))]
    best = None
    for s in starts:
        for meth, opt in (("Nelder-Mead", dict(maxiter=20000, maxfev=40000,
                                               xatol=1e-9, fatol=1e-9)),
                          ("Powell",      dict(maxiter=20000, maxfev=40000,
                                               xtol=1e-9, ftol=1e-9))):
            try:
                r = optimize.minimize(fn, s, args=args, method=meth, options=opt)
            except Exception:
                continue
            if np.isfinite(r.fun) and (best is None or r.fun < best.fun - 1e-12):
                best = r
            if np.all(np.isfinite(r.x)):
                s = r.x                      # chain: Powell polishes the NM solution
    return best

def _hessian(fn, x, args, rel=1e-4):
    x = np.asarray(x, float); n = len(x)
    h = np.maximum(rel * np.abs(x), 1e-5)
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            ei = np.zeros(n); ei[i] = h[i]
            ej = np.zeros(n); ej[j] = h[j]
            f1 = fn(x + ei + ej, *args); f2 = fn(x + ei - ej, *args)
            f3 = fn(x - ei + ej, *args); f4 = fn(x - ei - ej, *args)
            if max(f1, f2, f3, f4) >= BIG_NLL:      # perturbation left the support
                return None
            H[i, j] = H[j, i] = (f1 - f2 - f3 + f4) / (4 * h[i] * h[j])
    return H

def _stderr(fn, x, args):
    H = _hessian(fn, x, args)
    if H is None:
        return np.full(len(x), np.nan)
    try:
        C = np.linalg.inv(H)
    except np.linalg.LinAlgError:
        C = np.linalg.pinv(H)
    d = np.asarray(np.diag(C), float)
    ok = np.isfinite(d) & (d > 0)
    return np.where(ok, np.sqrt(np.where(ok, d, 1.0)), np.nan)   # NaN, not sqrt(abs(.))

def _init_vector(dist, t, ns, m, loc_max):
    try:
        init = dist.fit(t, floc=0)
        shapes = list(init[:ns]); sc = float(init[-1])
    except Exception:
        shapes = [1.0] * ns; sc = float(np.mean(t))
    x0 = shapes + [np.log(max(sc, 1e-6))] + [0.0] * m
    if loc_max is not None:
        x0 = x0 + [-4.0]                            # start with a small threshold
    return np.array(x0, float)

def fit_aft(t, Z, dist_name, warm=None, n_starts=None):
    """Fit the AFT model. Returns a dict, or None if it could not be fitted."""
    n_starts = N_STARTS_SEARCH if n_starts is None else n_starts
    dist = getattr(stats, dist_name)
    ns, m = dist.numargs, Z.shape[1]
    loc_max = None if LOC_MODE == "zero" else float(np.min(t)) * 0.999
    npar = ns + 1 + m + (0 if loc_max is None else 1)
    x0 = np.asarray(warm, float) if (warm is not None and len(warm) == npar) \
         else _init_vector(dist, t, ns, m, loc_max)
    args = (t, Z, dist, ns, loc_max)
    r = _optimise(_nll, x0, args, n_starts)
    if r is None or not np.isfinite(r.fun) or r.fun >= BIG_NLL:
        return None
    return dict(dist=dist_name, ns=ns, m=m, loc_max=loc_max, x=np.asarray(r.x, float),
                ll=-float(r.fun), k=npar, args=args,
                converged=bool(getattr(r, "success", False)))

# --- warm-start helpers: keep loc_raw at the tail when adding/removing a beta --
def _extend(x, ns, m_old):
    idx = ns + 1 + m_old
    return np.concatenate([x[:idx], [0.0], x[idx:]])

def _shrink(x, ns, j):
    idx = ns + 1 + j
    return np.concatenate([x[:idx], x[idx + 1:]])

# ---------------------------------------------------------- Cox-Snell --------
def cox_snell(t, Z, fit):
    dist = getattr(stats, fit["dist"]); ns, m = fit["ns"], fit["m"]
    shapes, b0, betas, loc = _unpack(fit["x"], m, ns, fit["loc_max"])
    lin = b0 + (Z @ betas if m else 0.0)
    scale = np.exp(np.clip(lin, -ETA_CLIP, ETA_CLIP))
    F = np.clip(dist.cdf(t, *shapes, loc=loc, scale=scale), 1e-12, 1 - 1e-12)
    cs = -np.log(1 - F)
    D, p = stats.kstest(cs, "expon")
    n = len(cs); q = stats.expon.ppf((np.arange(1, n + 1) - 0.5) / n)
    cs_s = np.sort(cs)
    qq = float(np.corrcoef(cs_s, q)[0, 1])
    u = np.clip(stats.expon.cdf(cs_s), 1e-12, 1 - 1e-12)
    i = np.arange(1, n + 1)
    ad = float(-n - np.sum((2 * i - 1) * (np.log(u) + np.log(1 - u[::-1]))) / n)
    return cs, float(D), float(p), qq, ad, scale, shapes, loc

def cox_snell_boot_p(t, Z, fit, D_obs, n_boot, seed=RNG_SEED):
    """Parametric-bootstrap p for the Cox-Snell KS statistic (parameters estimated)."""
    if not n_boot:
        return np.nan
    dist = getattr(stats, fit["dist"]); ns, m = fit["ns"], fit["m"]
    shapes, b0, betas, loc = _unpack(fit["x"], m, ns, fit["loc_max"])
    lin = b0 + (Z @ betas if m else 0.0)
    scale = np.exp(np.clip(lin, -ETA_CLIP, ETA_CLIP))
    rg = np.random.default_rng(seed)
    cnt = done = 0
    for _ in range(n_boot):
        try:
            tb = dist.rvs(*shapes, loc=loc, scale=scale, size=len(t), random_state=rg)
            ok = np.isfinite(tb) & (tb > 0)
            if ok.sum() < max(10, len(t) // 2):
                continue
            fb = fit_aft(tb[ok], Z[ok], fit["dist"], warm=fit["x"], n_starts=1)
            if fb is None:
                continue
            _, Db, _, _, _, _, _, _ = cox_snell(tb[ok], Z[ok], fb)
            cnt += int(Db >= D_obs); done += 1
        except Exception:
            continue
    return round((cnt + 1) / (done + 1), 4) if done else np.nan

# ------------------------------------------------------------- design --------
def build_design(g, cov_names):
    """Master design for one pair: ONE row set shared by every model in that pair,
    so the whole stepwise path and the null are strictly nested and comparable."""
    cols = [COVARIATES[c]["col"] for c in cov_names]
    sub = g[[OUTCOME] + cols].apply(pd.to_numeric, errors="coerce")
    n_before = len(sub)
    sub = sub.dropna()
    sub = sub[sub[OUTCOME] > 0]
    t = sub[OUTCOME].values.astype(float)
    Z, scalers, used = [], {}, []
    for c in cov_names:
        meta = COVARIATES[c]
        x = sub[meta["col"]].values.astype(float)
        if meta["type"] == "cont":
            sd = float(np.std(x, ddof=1))
            if sd <= 1e-12:
                continue
            Z.append((x - x.mean()) / sd); scalers[c] = sd
        else:
            if len(np.unique(x)) != 2 or min((x == 0).sum(), (x == 1).sum()) < MIN_BIN_GROUP:
                continue
            Z.append(x); scalers[c] = 1.0
        used.append(c)
    Zm = np.column_stack(Z) if Z else np.empty((len(t), 0))
    return t, Zm, scalers, used, n_before - len(sub)

def fmt_p(p):
    if p is None or (isinstance(p, float) and not np.isfinite(p)):
        return "n/a"
    return "<0.001" if float(p) < 1e-3 else round(float(p), 4)

def stars(p):
    if p is None or (isinstance(p, float) and not np.isfinite(p)):
        return ""
    p = float(p)
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

print("AFT machinery ready.")

AFT machinery ready.


In [5]:
# =============================================================================
# Cell 5 — Stepwise covariate selection: FORWARD entry with BACKWARD fall-back
#
# The old notebook did pure forward selection and never revisited a covariate
# once it was in. With target speed and speed difference both present (rho ~ 0.22,
# and speed difference is a linear function of target and leading speed) that is
# exactly the situation where an early entrant becomes redundant after a later one
# is added -- the fall-back step is what removes it.
#
# Every trial model is warm-started from its parent, so LR statistics along the
# path are non-negative by construction and the ENTER / STAY decisions are made on
# genuinely nested comparisons.  All models within a pair share one row set
# (built once in Cell 6), so the whole path is strictly nested.
# =============================================================================
def _sub(Zfull, used, names):
    if not names:
        return np.empty((Zfull.shape[0], 0))
    return Zfull[:, [used.index(c) for c in names]]

def stepwise_select(t, Zfull, used, dist_name, f_null):
    """Return (selected, final_fit, path_rows, note).

    `f_null` is the intercept-only fit supplied by the caller so that the path and
    the joint LR test reported later are anchored to the SAME null likelihood.
    """
    ns = getattr(stats, dist_name).numargs
    path = []
    f_cur = f_null
    if f_cur is None:
        return None, None, path, "null model did not fit"
    selected = []
    path.append(dict(step=0, action="start", Covariate="(intercept only)",
                     dAIC=np.nan, LR_p="n/a", set_after="none",
                     AIC=round(2 * f_cur["k"] - 2 * f_cur["ll"], 2)))

    visited = {frozenset()}
    note = "converged: no further covariate met the entry rule"
    step = 0
    while step < MAX_STEPS:
        step += 1

        # ------------------------------------------------ FORWARD entry -------
        best = None
        for cand in [c for c in used if c not in selected]:
            names = selected + [cand]
            fc = fit_aft(t, _sub(Zfull, used, names), dist_name,
                         warm=_extend(f_cur["x"], ns, len(selected)))
            if fc is None:
                continue
            LR = max(2.0 * (fc["ll"] - f_cur["ll"]), 0.0)
            p = float(stats.chi2.sf(LR, 1))
            dAIC = LR - 2.0
            if dAIC >= DAIC_ENTER and p < ALPHA_ENTER and (best is None or dAIC > best["dAIC"]):
                best = dict(cov=cand, dAIC=dAIC, p=p, fit=fc, names=names)
        if best is None:
            break

        selected = best["names"]; f_cur = best["fit"]
        path.append(dict(step=step, action="ENTER", Covariate=best["cov"],
                         dAIC=round(best["dAIC"], 2), LR_p=fmt_p(best["p"]),
                         set_after=", ".join(selected),
                         AIC=round(2 * f_cur["k"] - 2 * f_cur["ll"], 2)))

        # ------------------------------------- BACKWARD fall-back -------------
        if USE_BACKWARD:
            changed = True
            while changed and len(selected) >= 2:
                changed = False
                for j, cov in enumerate(selected):
                    names = selected[:j] + selected[j + 1:]
                    fr = fit_aft(t, _sub(Zfull, used, names), dist_name,
                                 warm=_shrink(f_cur["x"], ns, j))
                    if fr is None:
                        continue
                    LR = max(2.0 * (f_cur["ll"] - fr["ll"]), 0.0)
                    p = float(stats.chi2.sf(LR, 1))
                    dAIC = LR - 2.0
                    if not (dAIC >= DAIC_STAY and p < ALPHA_STAY):
                        selected = names; f_cur = fr; changed = True
                        path.append(dict(step=step, action="DROP (fall-back)",
                                         Covariate=cov, dAIC=round(dAIC, 2),
                                         LR_p=fmt_p(p),
                                         set_after=", ".join(selected) if selected else "none",
                                         AIC=round(2 * f_cur["k"] - 2 * f_cur["ll"], 2)))
                        break

        key = frozenset(selected)
        if key in visited:
            note = "stopped: selection cycled back to a set already visited"
            break
        visited.add(key)
    else:
        note = f"stopped: hit MAX_STEPS={MAX_STEPS}"

    # final refit with more starts, warm-started from the path solution
    f_fin = fit_aft(t, _sub(Zfull, used, selected), dist_name,
                    warm=f_cur["x"], n_starts=N_STARTS_FINAL)
    if f_fin is None or f_fin["ll"] < f_cur["ll"]:
        f_fin = f_cur
    return selected, f_fin, path, note

print("Stepwise selector ready (forward entry + backward fall-back, cycle-guarded).")

Stepwise selector ready (forward entry + backward fall-back, cycle-guarded).


In [6]:
# =============================================================================
# Cell 6 — Build + verify each pair: stepwise selection under each ranked
#          distribution, with the fall-back cascade
#
#   for each pair:
#       one master row set  ->  every model in the pair is strictly nested
#       for each distribution in Table-3 AIC order (max N_CASCADE):
#           fit the null  ->  stepwise forward/backward  ->  Cox-Snell check
#           CASCADE_RULE == "first_pass": stop at the first that passes
#           CASCADE_RULE == "best_pass" : try all, keep the lowest-AIC passer
#       nothing passes -> keep the lowest-AIC attempt, flagged UNVERIFIED
#
# Every attempt is logged, not just the winner. The cascade is a search over
# distributions, so the paper has to show what was searched.
# =============================================================================
attempt_rows, coef_rows, gof_rows, path_rows, cs_store, chosen_store = [], [], [], [], {}, {}

for pair in PAIRS:
    g = df[df[STRATUM] == pair]
    pool = CANDIDATES.get(pair, [])
    t, Zfull, scalers, used, ndrop = build_design(g, pool)
    n = len(t)
    order = RANKING[pair][:N_CASCADE]
    not_estimable = [c for c in pool if c not in used]
    print(f"\n{pair}  n={n}  candidates={used}"
          + (f"  (not estimable: {not_estimable})" if not_estimable else ""))

    attempts = []
    for di, dname in enumerate(order):
        f_null = fit_aft(t, np.empty((n, 0)), dname, n_starts=N_STARTS_FINAL)
        if f_null is None:
            attempt_rows.append(dict(Pair=pair, rank=di + 1, Distribution=dname,
                                     status="null model failed"))
            continue
        selected, f_fin, path, note = stepwise_select(t, Zfull, used, dname, f_null)
        if f_fin is None:
            attempt_rows.append(dict(Pair=pair, rank=di + 1, Distribution=dname,
                                     status="stepwise failed: " + str(note)))
            continue

        Zs = _sub(Zfull, used, selected)
        cs, D, p_ks, qq, ad, scale_i, shapes, loc = cox_snell(t, Zs, f_fin)
        p_gate = (cox_snell_boot_p(t, Zs, f_fin, D, N_BOOT_CS,
                                   seed=RNG_SEED + di * 101 + (abs(hash(pair)) % 997))
                  if CS_GATE == "ks_bootstrap" else p_ks)
        passed = bool(np.isfinite(p_gate) and p_gate >= CS_ALPHA and f_fin["converged"])

        LR = max(2.0 * (f_fin["ll"] - f_null["ll"]), 0.0)
        p_joint = float(stats.chi2.sf(LR, len(selected))) if selected else np.nan
        AIC  = 2 * f_fin["k"]  - 2 * f_fin["ll"]
        AIC0 = 2 * f_null["k"] - 2 * f_null["ll"]

        rec = dict(Pair=pair, rank=di + 1, Distribution=dname, n=n,
                   Selected=", ".join(selected) if selected else "none",
                   n_cov=len(selected), n_params=f_fin["k"],
                   LogLik=round(f_fin["ll"], 3), AIC=round(AIC, 2),
                   BIC=round(f_fin["k"] * np.log(n) - 2 * f_fin["ll"], 2),
                   dAIC_vs_null=round(AIC0 - AIC, 2),
                   joint_LR=round(LR, 3), joint_df=len(selected), joint_p=fmt_p(p_joint),
                   CoxSnell_KS_D=round(D, 4), CoxSnell_KS_p=fmt_p(p_ks),
                   CoxSnell_gate_p=fmt_p(p_gate), CoxSnell_AD=round(ad, 3),
                   QQ_corr=round(qq, 4),
                   converged="Yes" if f_fin["converged"] else "check",
                   passed="Yes" if passed else "No", status="ok")
        attempt_rows.append(rec)
        attempts.append(dict(rec=rec, di=di, dname=dname, selected=selected,
                             fit=f_fin, null=f_null, Zs=Zs, cs=cs, path=path,
                             note=note, passed=passed, AIC=AIC, AIC0=AIC0,
                             D=D, p_ks=p_ks, p_gate=p_gate, qq=qq, ad=ad,
                             LR=LR, p_joint=p_joint, shapes=shapes, loc=loc))
        sel_txt  = ", ".join(selected) if selected else "-"
        gate_txt = f"{p_gate:.3f}" if np.isfinite(p_gate) else "n/a"
        flag_txt = "PASS" if passed else "fail"
        print(f"   rank{di+1:>2} {dname:12s} sel=[{sel_txt}]  AIC={AIC:8.2f}  "
              f"CS_KS_p={p_ks:.3f}  gate={gate_txt}  -> {flag_txt}")

        if CASCADE_RULE == "first_pass" and passed:
            break

    if not attempts:
        print("   !! no distribution could be fitted for this pair")
        continue

    passers = [a for a in attempts if a["passed"]]
    if passers:
        chosen = passers[0] if CASCADE_RULE == "first_pass" else min(passers, key=lambda a: a["AIC"])
        verdict = ("primary (rank-1 distribution)" if chosen["di"] == 0 else
                   f"FALL-BACK -> rank-{chosen['di']+1} {chosen['dname']} "
                   f"(rank-1 {order[0]} failed the Cox-Snell check)")
    else:
        chosen = min(attempts, key=lambda a: a["AIC"])
        verdict = (f"UNVERIFIED - no distribution passed; best AIC was "
                   f"rank-{chosen['di']+1} {chosen['dname']}")

    chosen_store[pair] = chosen
    cs_store[pair] = chosen["cs"]
    f_fin, f_null, selected = chosen["fit"], chosen["null"], chosen["selected"]
    ns = f_fin["ns"]
    se = _stderr(_nll, f_fin["x"], f_fin["args"])

    for r in chosen["path"]:
        path_rows.append(dict(Pair=pair, Distribution=chosen["dname"], **r))

    for i, cov in enumerate(selected):
        meta = COVARIATES[cov]; sd = scalers.get(cov, 1.0)
        b_std = float(f_fin["x"][ns + 1 + i]); s_std = float(se[ns + 1 + i])
        if meta["type"] == "cont":
            b_raw, u, div = b_std / sd, meta["unit"], sd
        else:
            b_raw, u, div = b_std, 1.0, 1.0
        eff = (np.exp(b_raw * u) - 1) * 100
        if np.isfinite(s_std) and s_std > 0:
            lo = (np.exp((b_std - 1.96 * s_std) / div * u) - 1) * 100
            hi = (np.exp((b_std + 1.96 * s_std) / div * u) - 1) * 100
            z = b_std / s_std
            pw = float(2 * stats.norm.sf(abs(z)))
        else:
            lo = hi = z = pw = np.nan
        mg = MARGINAL.get(pair, {}).get(cov, {})
        coef_rows.append(dict(
            Pair=pair, N=n, Distribution=chosen["dname"], Covariate=cov,
            Type=meta["type"], Unit=meta["label"],
            coef_std=round(b_std, 6), SE_std=round(s_std, 6) if np.isfinite(s_std) else np.nan,
            coef_per_unit=round(b_raw, 6),
            Effect=f"{eff:+.2f}%  {meta['label']}",
            CI_low_pct=round(lo, 2) if np.isfinite(lo) else np.nan,
            CI_high_pct=round(hi, 2) if np.isfinite(hi) else np.nan,
            Wald_z=round(z, 3) if np.isfinite(z) else np.nan,
            Wald_p=fmt_p(pw), sig=stars(pw),
            marginal_effect_T4=mg.get("marginal_effect"),
            marginal_dAIC_T4=mg.get("marginal_dAIC"),
            marginal_LR_p_T4=mg.get("marginal_LR_p")))
    if not selected:
        coef_rows.append(dict(Pair=pair, N=n, Distribution=chosen["dname"],
                              Covariate="(none retained)", Type="", Unit="",
                              Effect="intercept-only distribution"))

    shp = ", ".join(f"{v:.4f}" for v in chosen["shapes"]) if len(chosen["shapes"]) else "-"
    gof_rows.append(dict(
        Pair=pair, N=n, Distribution=chosen["dname"],
        rank_in_Table3=chosen["di"] + 1, cascade_verdict=verdict,
        Final_covariates=", ".join(selected) if selected else "none (decoupled)",
        n_candidates_offered=len(used), n_retained=len(selected),
        shape_params=shp, loc=round(chosen["loc"], 4),
        scale_intercept_b0=round(float(f_fin["x"][ns]), 5),
        LogLik=round(f_fin["ll"], 3), n_params=f_fin["k"],
        AIC=round(chosen["AIC"], 2),
        BIC=round(f_fin["k"] * np.log(n) - 2 * f_fin["ll"], 2),
        dAIC_vs_null=round(chosen["AIC0"] - chosen["AIC"], 2),
        joint_LR=round(chosen["LR"], 3), joint_df=len(selected),
        joint_p=fmt_p(chosen["p_joint"]),
        CoxSnell_KS_D=round(chosen["D"], 4), CoxSnell_KS_p=fmt_p(chosen["p_ks"]),
        CoxSnell_gate_p=fmt_p(chosen["p_gate"]), CoxSnell_AD=round(chosen["ad"], 3),
        QQ_corr=round(chosen["qq"], 4),
        fit_ok="Yes" if chosen["passed"] else "NO",
        converged="Yes" if f_fin["converged"] else "check",
        n_distributions_tried=len(attempts),
        selection_note=chosen["note"], n_dropped_missing=ndrop))

attempts_log = pd.DataFrame(attempt_rows)
coefficients = pd.DataFrame(coef_rows)
goodness     = pd.DataFrame(gof_rows)
sel_path     = pd.DataFrame(path_rows)
goodness


BTW_following_4W  n=250  candidates=['Speed_Difference', 'Target_Speed_km/hr', 'Site', 'Off_centeredness', 'Occupancy', 'Flow_pcu/hr/m']
   rank 1 gengamma     sel=[Speed_Difference, Target_Speed_km/hr, Off_centeredness]  AIC=  564.89  CS_KS_p=0.963  gate=0.963  -> PASS

BTW_following_MT_3W  n=186  candidates=['Target_Speed_km/hr', 'Speed_Difference', 'Site', 'Off_centeredness', 'Occupancy', 'Flow_pcu/hr/m']
   rank 1 gamma        sel=[Target_Speed_km/hr, Speed_Difference]  AIC=  387.72  CS_KS_p=0.145  gate=0.145  -> PASS

BTW_following_NMT_3W  n=119  candidates=['Target_Speed_km/hr', 'Site', 'Flow_pcu/hr/m', 'Off_centeredness', 'Speed_Difference', 'Occupancy']
   rank 1 gamma        sel=[Target_Speed_km/hr]  AIC=  293.91  CS_KS_p=0.887  gate=0.887  -> PASS

PR_following_MT_3W  n=104  candidates=['Target_Speed_km/hr', 'Occupancy', 'Speed_Difference', 'Site', 'Off_centeredness', 'Flow_pcu/hr/m']
   rank 1 weibull_min  sel=[Target_Speed_km/hr, Occupancy]  AIC=  257.14  CS_KS_p=0.681  ga

,Pair,N,Distribution,rank_in_Table3,cascade_verdict,Final_covariates,n_candidates_offered,n_retained,shape_params,loc,...,CoxSnell_KS_D,CoxSnell_KS_p,CoxSnell_gate_p,CoxSnell_AD,QQ_corr,fit_ok,converged,n_distributions_tried,selection_note,n_dropped_missing
0,BTW_following_4W,250,gengamma,1,primary (rank-1 distribution),"Speed_Difference, Target_Speed_km/hr, Off_cent...",6,3,"2.1844, 2.1113",0.0,...,0.0311,0.9629,0.9629,0.290,0.9979,Yes,Yes,1,converged: no further covariate met the entry ...,0
1,BTW_following_MT_3W,186,gamma,1,primary (rank-1 distribution),"Target_Speed_km/hr, Speed_Difference",6,2,6.5469,0.0,...,0.0831,0.1449,0.1449,1.390,0.9845,Yes,Yes,1,converged: no further covariate met the entry ...,0
2,BTW_following_NMT_3W,119,gamma,1,primary (rank-1 distribution),Target_Speed_km/hr,6,1,4.7569,0.0,...,0.0521,0.8868,0.8868,0.441,0.9896,Yes,Yes,1,converged: no further covariate met the entry ...,0
3,PR_following_MT_3W,104,weibull_min,1,primary (rank-1 distribution),"Target_Speed_km/hr, Occupancy",6,2,3.9924,0.0,...,0.0689,0.6813,0.6813,0.339,0.9914,Yes,Yes,1,converged: no further covariate met the entry ...,0
4,BTW_following_MT_2W,73,invgauss,1,primary (rank-1 distribution),"Speed_Difference, Site",6,2,0.2015,0.0,...,0.0968,0.4718,0.4718,1.001,0.9772,Yes,Yes,1,converged: no further covariate met the entry ...,0
5,PR_following_NMT_3W,49,weibull_min,1,primary (rank-1 distribution),none (decoupled),6,0,3.0628,0.0,...,0.0912,0.7758,0.7758,0.405,0.9830,Yes,Yes,1,converged: no further covariate met the entry ...,0
6,PR_following_4W,43,weibull_min,1,primary (rank-1 distribution),none (decoupled),6,0,3.1566,0.0,...,0.1426,0.3152,0.3152,0.797,0.9798,Yes,Yes,1,converged: no further covariate met the entry ...,0
7,BTW_following_NMT_2W,41,invgauss,1,primary (rank-1 distribution),Speed_Difference,6,1,0.1983,0.0,...,0.1035,0.7331,0.7331,0.414,0.9671,Yes,Yes,1,converged: no further covariate met the entry ...,0


In [7]:
# =============================================================================
# Cell 7 — Write all tables.  Output is 07_final_models_verified.xlsx, NOT 06_*,
#          because the master pipeline already owns 06_compound_models.xlsx and
#          the old notebook silently overwrote it.
# =============================================================================
if not len(goodness):
    raise RuntimeError("No pair produced a model -- check the reconciliation warnings "
                       "printed by Cell 3 before running this cell.")

pub = (coefficients[coefficients["Covariate"] != "(none retained)"].copy()
       if len(coefficients) else pd.DataFrame())
if len(pub):
    pub["Estimate (95% CI)"] = pub.apply(
        lambda r: (f"{r['Effect'].split('  ')[0]} [{r['CI_low_pct']:+.1f}, {r['CI_high_pct']:+.1f}]"
                   if pd.notna(r.get("CI_low_pct")) else r["Effect"]), axis=1)
    pub = pub[["Pair", "Distribution", "N", "Covariate", "Unit",
               "Estimate (95% CI)", "Wald_p", "sig"]]

# marginal (Table 4, unadjusted) vs conditional (final model, adjusted)
comp_rows = []
for pair in PAIRS:
    ch = chosen_store.get(pair)
    if ch is None:
        continue
    keep = set(ch["selected"])
    for cov, mg in MARGINAL.get(pair, {}).items():
        if cov not in COVARIATES:
            continue
        sub = coefficients[(coefficients.Pair == pair) & (coefficients.Covariate == cov)]
        comp_rows.append(dict(
            Pair=pair, Covariate=cov,
            marginal_effect=mg.get("marginal_effect"),
            marginal_dAIC=mg.get("marginal_dAIC"),
            marginal_LR_p=mg.get("marginal_LR_p"),
            retained="Yes" if cov in keep else "No",
            adjusted_effect=(sub["Effect"].iloc[0] if len(sub) else ""),
            adjusted_p=(sub["Wald_p"].iloc[0] if len(sub) else ""),
            interpretation=("retained after adjustment" if cov in keep else
                            "marginal effect did not survive conditional entry")))
marg_vs_cond = pd.DataFrame(comp_rows)

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as xl:
    goodness.to_excel(xl,     sheet_name="Model_fit_GoF",    index=False)
    if len(pub):
        pub.to_excel(xl,      sheet_name="Publication_table", index=False)
    coefficients.to_excel(xl, sheet_name="Coefficients",     index=False)
    sel_path.to_excel(xl,     sheet_name="Selection_path",   index=False)
    attempts_log.to_excel(xl, sheet_name="Cascade_attempts", index=False)
    if len(marg_vs_cond):
        marg_vs_cond.to_excel(xl, sheet_name="Marginal_vs_adjusted", index=False)
print("Saved:", OUT_XLSX)

unver = goodness.loc[goodness["fit_ok"] == "NO", "Pair"].tolist()
fell  = goodness.loc[goodness["rank_in_Table3"] > 1, "Pair"].tolist()
print("Fell back to a lower-ranked distribution:", fell if fell else "none")
print("UNVERIFIED (no distribution passed Cox-Snell):", unver if unver else "none")
if CS_GATE == "ks_analytic":
    print("\nNOTE: the Cox-Snell gate used the analytic KS p-value. Because the "
          "parameters were estimated from the same sample it is biased upward, so "
          "the gate is LENIENT and the cascade will rarely fire. Set "
          "CS_GATE='ks_bootstrap' for the version to report.")
coefficients

Saved: D:\Headway\Tables\07_final_models_verified.xlsx
Fell back to a lower-ranked distribution: none
UNVERIFIED (no distribution passed Cox-Snell): none

NOTE: the Cox-Snell gate used the analytic KS p-value. Because the parameters were estimated from the same sample it is biased upward, so the gate is LENIENT and the cascade will rarely fire. Set CS_GATE='ks_bootstrap' for the version to report.


,Pair,N,Distribution,Covariate,Type,Unit,coef_std,SE_std,coef_per_unit,Effect,CI_low_pct,CI_high_pct,Wald_z,Wald_p,sig,marginal_effect_T4,marginal_dAIC_T4,marginal_LR_p_T4
0,BTW_following_4W,250,gengamma,Speed_Difference,cont,per +1 km/h,-0.115190,0.020734,-0.023970,-2.37% per +1 km/h,-3.19,-1.54,-5.556,<0.001,***,-2.64% per +1 km/h,30.02,<0.001
1,BTW_following_4W,250,gengamma,Target_Speed_km/hr,cont,per +1 km/h,-0.105616,0.019765,-0.019364,-1.92% per +1 km/h,-2.61,-1.22,-5.344,<0.001,***,-2.10% per +1 km/h,26.80,<0.001
2,BTW_following_4W,250,gengamma,Off_centeredness,bin,True vs False,-0.110429,0.048290,-0.110429,-10.45% True vs False,-18.54,-1.57,-2.287,0.0222,*,-11.26% (True vs False),3.03,0.0249
3,BTW_following_MT_3W,186,gamma,Target_Speed_km/hr,cont,per +1 km/h,-0.153315,0.030462,-0.026956,-2.66% per +1 km/h,-3.68,-1.63,-5.033,<0.001,***,-3.51% per +1 km/h,38.74,<0.001
4,BTW_following_MT_3W,186,gamma,Speed_Difference,cont,per +1 km/h,-0.133355,0.031509,-0.026436,-2.61% per +1 km/h,-3.79,-1.41,-4.232,<0.001,***,-3.77% per +1 km/h,33.38,<0.001
5,BTW_following_NMT_3W,119,gamma,Target_Speed_km/hr,cont,per +1 km/h,-0.101590,0.040136,-0.023321,-2.31% per +1 km/h,-4.05,-0.52,-2.531,0.0114,*,-2.31% per +1 km/h,4.10,0.0135
6,PR_following_MT_3W,104,weibull_min,Target_Speed_km/hr,cont,per +1 km/h,-0.096402,0.022448,-0.026837,-2.65% per +1 km/h,-3.83,-1.45,-4.294,<0.001,***,-2.97% per +1 km/h,16.08,<0.001
7,PR_following_MT_3W,104,weibull_min,Occupancy,bin,True vs False,-0.143710,0.053236,-0.143710,-13.39% True vs False,-21.97,-3.86,-2.699,0.0069,**,-17.03% (True vs False),9.22,<0.001
8,BTW_following_MT_2W,73,invgauss,Speed_Difference,cont,per +1 km/h,-0.190652,0.053783,-0.036564,-3.59% per +1 km/h,-5.52,-1.62,-3.545,<0.001,***,-2.72% per +1 km/h,4.99,0.0082
9,BTW_following_MT_2W,73,invgauss,Site,bin,Tikatuli vs Shahjahanpur,0.268950,0.107237,0.268950,+30.86% Tikatuli vs Shahjahanpur,6.05,61.47,2.508,0.0121,*,+14.30% (Tikatuli vs Shahjahanpur),-0.51,0.2224


In [8]:
# =============================================================================
# Cell 8 — Cox-Snell Q-Q charts embedded natively in the workbook (no matplotlib)
#   Sheet names are sanitised and de-duplicated: Excel forbids  : \ / ? * [ ]
#   and caps titles at 31 characters, so two long pair names could previously
#   collide and raise on create_sheet.
# =============================================================================
from openpyxl import load_workbook
from openpyxl.chart import ScatterChart, Reference, Series

def _safe_sheet(name, taken):
    s = re.sub(r"[:\\/?*\[\]]", "_", str(name))[:31]
    base, i = s, 1
    while s.lower() in taken:
        suf = f"~{i}"
        s = base[:31 - len(suf)] + suf
        i += 1
    taken.add(s.lower())
    return s

wb = load_workbook(OUT_XLSX)
taken = {ws.title.lower() for ws in wb.worksheets}

for pair, cs in cs_store.items():
    cs_sorted = np.sort(np.asarray(cs, float))
    n = len(cs_sorted)
    if n < 3:
        continue
    theo = stats.expon.ppf((np.arange(1, n + 1) - 0.5) / n)
    ws = wb.create_sheet(_safe_sheet("qq_" + str(pair), taken))
    ws["A1"], ws["B1"], ws["D1"], ws["E1"] = "theoretical", "cox_snell", "ref_x", "ref_y"
    for i in range(n):
        ws.cell(row=i + 2, column=1, value=float(theo[i]))
        ws.cell(row=i + 2, column=2, value=float(cs_sorted[i]))
    mx = float(max(theo.max(), cs_sorted.max()))
    ws["D2"], ws["E2"], ws["D3"], ws["E3"] = 0, 0, mx, mx      # 45-degree line

    ch = ScatterChart()
    ch.title = f"Cox-Snell Q-Q: {pair}"
    ch.x_axis.title = "Exponential(1) quantile"
    ch.y_axis.title = "Cox-Snell residual"
    ch.x_axis.delete = False
    ch.y_axis.delete = False
    s_pts = Series(Reference(ws, min_col=2, min_row=1, max_row=n + 1),
                   Reference(ws, min_col=1, min_row=2, max_row=n + 1),
                   title_from_data=True)
    s_pts.marker.symbol = "circle"
    s_pts.graphicalProperties.line.noFill = True
    s_ref = Series(Reference(ws, min_col=5, min_row=1, max_row=3),
                   Reference(ws, min_col=4, min_row=2, max_row=3),
                   title_from_data=True)
    ch.series.append(s_pts)
    ch.series.append(s_ref)
    ch.height, ch.width = 9, 12
    ws.add_chart(ch, "G2")

wb.save(OUT_XLSX)
print(f"Embedded {len(cs_store)} Cox-Snell Q-Q charts into:", OUT_XLSX)

Embedded 8 Cox-Snell Q-Q charts into: D:\Headway\Tables\07_final_models_verified.xlsx


In [9]:
# =============================================================================
# Cell 9 — Publication figures into the Graphics folder (optional, MAKE_PNG)
#   fig07a  Cox-Snell Q-Q panel, one axis per pair
#   fig07b  forest plot of the retained, mutually adjusted covariate effects
# =============================================================================
if MAKE_PNG and len(cs_store):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9,
                         "axes.grid": True, "grid.alpha": 0.25, "savefig.bbox": "tight"})

    pairs_plot = [p for p in PAIRS if p in cs_store]
    ncol = 3
    nrow = int(np.ceil(len(pairs_plot) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 2.8 * nrow), squeeze=False)
    axes = axes.ravel()
    for ax, pair in zip(axes, pairs_plot):
        cs_s = np.sort(np.asarray(cs_store[pair], float))
        n = len(cs_s)
        q = stats.expon.ppf((np.arange(1, n + 1) - 0.5) / n)
        row = goodness.loc[goodness["Pair"] == pair]
        dn = row["Distribution"].iloc[0] if len(row) else ""
        pk = row["CoxSnell_KS_p"].iloc[0] if len(row) else ""
        ax.plot(q, cs_s, "o", ms=2.6, color="#4C72B0", alpha=.75)
        lim = [0, float(max(q.max(), cs_s.max()))]
        ax.plot(lim, lim, "--", color="#C44E52", lw=1)
        ax.set_title(f"{pair}\n{dn}  (KS p={pk})", fontsize=8)
        ax.set_xlabel("Exponential(1) quantile")
        ax.set_ylabel("Cox-Snell residual")
    for ax in axes[len(pairs_plot):]:
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(os.path.join(GRAPHICS, "fig07a_coxsnell_qq.png"))
    plt.close(fig)
    print("Saved:", os.path.join(GRAPHICS, "fig07a_coxsnell_qq.png"))

    fp = coefficients.dropna(subset=["CI_low_pct"]).copy() if "CI_low_pct" in coefficients else pd.DataFrame()
    if len(fp):
        fp = fp.sort_values(["Pair", "Covariate"]).reset_index(drop=True)
        centre = [(np.exp(r.coef_per_unit * (COVARIATES[r.Covariate]["unit"]
                   if COVARIATES[r.Covariate]["type"] == "cont" else 1.0)) - 1) * 100
                  for r in fp.itertuples()]
        y = np.arange(len(fp))
        fig, ax = plt.subplots(figsize=(7.2, 0.34 * len(fp) + 2.0))
        ax.hlines(y, fp["CI_low_pct"], fp["CI_high_pct"], color="#4C72B0", lw=2)
        ax.plot(centre, y, "o", color="#C44E52", ms=4)
        ax.axvline(0, color="k", lw=1)
        ax.set_yticks(y)
        ax.set_yticklabels([f"{r.Pair} | {r.Covariate}" for r in fp.itertuples()], fontsize=7)
        ax.invert_yaxis()
        ax.set_xlabel("% change in the scale parameter (\u2248 % change in median headway)")
        ax.set_title("Final stepwise-selected AFT models: adjusted effects with 95% CI")
        fig.savefig(os.path.join(GRAPHICS, "fig07b_forest_final.png"))
        plt.close(fig)
        print("Saved:", os.path.join(GRAPHICS, "fig07b_forest_final.png"))
else:
    print("MAKE_PNG is False or nothing to plot -- skipped.")

Saved: D:\Headway\Graphics\fig07a_coxsnell_qq.png
Saved: D:\Headway\Graphics\fig07b_forest_final.png


In [10]:
# =============================================================================
# Cell 10 — Provenance stamp for the methods section
# =============================================================================
import scipy
prov = pd.DataFrame([
    dict(Item="run timestamp", Value=datetime.now().strftime("%Y-%m-%d %H:%M:%S")),
    dict(Item="python / numpy / scipy / pandas",
         Value=f"{platform.python_version()} / {np.__version__} / {scipy.__version__} / {pd.__version__}"),
    dict(Item="RNG seed", Value=RNG_SEED),
    dict(Item="data", Value=DATA_PATH),
    dict(Item="distribution ranking from", Value=os.path.basename(EXCEL03)),
    dict(Item="covariate candidates from", Value=os.path.basename(EXCEL04)),
    dict(Item="candidate rule", Value=CANDIDATE_RULE),
    dict(Item="selection", Value=("forward entry + backward fall-back" if USE_BACKWARD
                                  else "forward entry only")),
    dict(Item="entry rule", Value=f"conditional dAIC >= {DAIC_ENTER} and LR p < {ALPHA_ENTER}"),
    dict(Item="stay rule",  Value=f"conditional dAIC >= {DAIC_STAY} and LR p < {ALPHA_STAY}"),
    dict(Item="cascade rule", Value=f"{CASCADE_RULE}, up to {N_CASCADE} distributions"),
    dict(Item="verification", Value=f"Cox-Snell residuals vs Exp(1), gate={CS_GATE}, "
                                    f"alpha={CS_ALPHA}, boot={N_BOOT_CS}"),
    dict(Item="loc mode", Value=LOC_MODE),
    dict(Item="pairs modelled", Value=len(goodness)),
    dict(Item="pairs that fell back", Value=int((goodness['rank_in_Table3'] > 1).sum())),
    dict(Item="pairs UNVERIFIED", Value=int((goodness['fit_ok'] == 'NO').sum())),
])

from openpyxl import load_workbook as _lwb
_wb = _lwb(OUT_XLSX)
_ws = _wb.create_sheet("Provenance", 0)
_ws.append(["Item", "Value"])
for _, r in prov.iterrows():
    _ws.append([str(r["Item"]), str(r["Value"])])
_wb.save(OUT_XLSX)

print("Provenance written into", OUT_XLSX)
print(goodness[["Pair", "Distribution", "rank_in_Table3", "Final_covariates",
                "AIC", "dAIC_vs_null", "CoxSnell_KS_p", "fit_ok",
                "cascade_verdict"]].to_string(index=False))
prov

Provenance written into D:\Headway\Tables\07_final_models_verified.xlsx
                Pair Distribution  rank_in_Table3                                       Final_covariates    AIC  dAIC_vs_null  CoxSnell_KS_p fit_ok               cascade_verdict
    BTW_following_4W     gengamma               1 Speed_Difference, Target_Speed_km/hr, Off_centeredness 564.89         58.74         0.9629    Yes primary (rank-1 distribution)
 BTW_following_MT_3W        gamma               1                   Target_Speed_km/hr, Speed_Difference 387.72         54.77         0.1449    Yes primary (rank-1 distribution)
BTW_following_NMT_3W        gamma               1                                     Target_Speed_km/hr 293.91          4.10         0.8868    Yes primary (rank-1 distribution)
  PR_following_MT_3W  weibull_min               1                          Target_Speed_km/hr, Occupancy 257.14         21.51         0.6813    Yes primary (rank-1 distribution)
 BTW_following_MT_2W     invgauss     

,Item,Value
0,run timestamp,2026-07-24 11:38:37
1,python / numpy / scipy / pandas,3.14.6 / 2.4.6 / 1.18.0 / 2.3.3
2,RNG seed,20250101
3,data,D:\Headway\data3.xlsx
4,distribution ranking from,03_distribution_fits.xlsx
5,covariate candidates from,04_covariate_screen.xlsx
6,candidate rule,all
7,selection,forward entry + backward fall-back
8,entry rule,conditional dAIC >= 2.0 and LR p < 0.05
9,stay rule,conditional dAIC >= 0.0 and LR p < 0.05
